# Agricultural Super-Resolution Project
## Complete Workflow for Plant Disease Diagnosis

This notebook guides you through training and evaluating super-resolution models for agricultural disease diagnosis.

**Estimated Time:** 1-2 weeks (mostly waiting for training)

**Prerequisites:**
- Google Drive with ~20GB free space
- PlantVillage dataset downloaded
- Basic Python knowledge
- Patience! 😊

## 🔧 Setup (Run Once)

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
import os
print("Google Drive mounted successfully!" if os.path.exists('/content/drive/MyDrive') else "ERROR: Drive not mounted")

### Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision matplotlib scikit-learn seaborn
print("✓ Dependencies installed")

### Step 3: Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected!")
    print("Go to: Runtime > Change runtime type > Hardware accelerator > GPU")

### Step 4: Setup Project Paths

In [ ]:
import sys
from pathlib import Path

# Define paths
PROJECT_ROOT = Path('/content/drive/MyDrive/AgriSR_Project')
CODE_DIR = PROJECT_ROOT / 'code'

# Add to Python path
sys.path.append(str(CODE_DIR))

# Verify files exist
required_files = [
    'generators.py',
    'discriminators.py', 
    'training_classes.py',
    'dataset_utils.py',
    'experiment_01_data_preparation.py',
    'experiment_02_train_baseline.py',
    'experiment_03_classification_impact.py'
]

print("Checking required files:")
for f in required_files:
    file_path = CODE_DIR / f
    status = "✓" if file_path.exists() else "✗"
    print(f"  {status} {f}")

missing = [f for f in required_files if not (CODE_DIR / f).exists()]
if missing:
    print(f"\n⚠️ Missing files: {missing}")
    print("Please upload them to:", CODE_DIR)
else:
    print("\n✓ All files present!")

---

## 📊 Experiment 01: Data Preparation

**Goal:** Create manageable subsets of the PlantVillage dataset

**Duration:** ~5 minutes

**Output:** 
- 25% stratified subset
- Dataset statistics
- Sampling comparison plots

In [ ]:
# Check if raw data exists
raw_data_dir = PROJECT_ROOT / 'data' / 'raw'

if not raw_data_dir.exists() or not list(raw_data_dir.iterdir()):
    print("⚠️ RAW DATA NOT FOUND!")
    print(f"\nPlease download PlantVillage dataset and extract to:")
    print(f"  {raw_data_dir}")
    print("\nDataset link: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset")
else:
    print("✓ Raw data found")
    print(f"  Classes: {len(list(raw_data_dir.iterdir()))}")

In [ ]:
# Run data preparation
from experiment_01_data_preparation import main

print("Running Experiment 01: Data Preparation\n" + "="*60)
main()
print("\n✓ Experiment 01 complete!")

In [ ]:
# View the sampling comparison plot
from IPython.display import Image, display

plot_path = PROJECT_ROOT / 'data' / 'processed' / 'stats' / 'sampling_comparison.png'
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print("Plot not found. Check if data preparation completed successfully.")

---

## 🎯 Experiment 02: Train SR Models

**Goal:** Train baseline super-resolution models

**Duration:** 12-18 hours total (across multiple sessions)

**Models:** SimpleGenerator, ResidualGenerator(4), ResidualGenerator(8)

⚠️ **IMPORTANT:** This will take many hours! Colab will disconnect, but the script handles it automatically. Just re-run this cell after reconnecting.

In [ ]:
# Check GPU again (important after reconnecting)
import torch
print(f"GPU: {'Available' if torch.cuda.is_available() else '❌ NOT AVAILABLE'}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Run training
from experiment_02_train_baseline import main

print("Running Experiment 02: Training SR Models\n" + "="*60)
print("This will take HOURS. Colab may disconnect - that's okay!")
print("Just reconnect and re-run this cell to resume.\n")

main()

print("\n✓ Training complete!")

In [ ]:
# Monitor training progress (run this to check status)
import json
from pathlib import Path

results_dir = PROJECT_ROOT / 'results'
models = ['simple', 'residual_4blocks', 'residual_8blocks']

print("Training Progress:")
print("="*60)

for model in models:
    metrics_file = results_dir / model / 'training_metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        
        last_epoch = metrics['epochs'][-1] if metrics['epochs'] else 0
        last_g_loss = metrics['g_losses'][-1] if metrics['g_losses'] else 'N/A'
        
        print(f"{model}:")
        print(f"  Epoch: {last_epoch}/50")
        print(f"  G_loss: {last_g_loss:.4f}" if isinstance(last_g_loss, float) else f"  G_loss: {last_g_loss}")
    else:
        print(f"{model}: Not started yet")
    print()

In [ ]:
# View training curves
from IPython.display import Image, display
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, model in enumerate(models):
    curve_path = results_dir / model / 'training_curves.png'
    if curve_path.exists():
        img = plt.imread(str(curve_path))
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(model)
    else:
        axes[idx].text(0.5, 0.5, 'Not available', 
                      ha='center', va='center')
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

---

## 📈 Experiment 03: Classification Impact

**Goal:** Evaluate if SR actually improves disease classification

**Duration:** ~2-3 hours

**Pipeline:**
1. Generate SR images with trained models
2. Train disease classifiers on LR vs SR images
3. Compare accuracy

⚠️ Make sure Experiment 02 is complete before running this!

In [ ]:
# Verify trained models exist
print("Checking for trained models:")
models_ready = True

for model in models:
    model_path = results_dir / model / 'generator_final.pth'
    status = "✓" if model_path.exists() else "✗"
    print(f"  {status} {model}")
    if not model_path.exists():
        models_ready = False

if not models_ready:
    print("\n⚠️ Some models missing. Complete Experiment 02 first!")
else:
    print("\n✓ All models ready!")

In [ ]:
# Run classification evaluation
from experiment_03_classification_impact import main

print("Running Experiment 03: Classification Impact\n" + "="*60)
main()
print("\n✓ Experiment 03 complete!")

In [ ]:
# View results
from IPython.display import Image, display

comparison_plot = PROJECT_ROOT / 'results' / 'classification' / 'sr_comparison.png'
if comparison_plot.exists():
    display(Image(filename=str(comparison_plot)))
else:
    print("Results plot not found.")

In [ ]:
# Print numerical results
import json

results_file = PROJECT_ROOT / 'results' / 'classification' / 'results.json'
if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    
    print("Classification Results:")
    print("="*60)
    
    lr_baseline = results['lr_baseline']['best_val_acc']
    print(f"LR Baseline: {lr_baseline:.2f}%")
    print()
    
    for model in models:
        if model in results:
            acc = results[model]['best_val_acc']
            improvement = acc - lr_baseline
            print(f"{model}:")
            print(f"  Accuracy: {acc:.2f}%")
            print(f"  Improvement: {improvement:+.2f}%")
            print()
else:
    print("Results file not found.")

---

## 📝 Analysis Questions

Now that you have results, answer these questions:

### 1. Overall Impact
- Does SR improve classification accuracy?
- By how much?
- Is the improvement statistically significant?

### 2. Model Comparison
- Which SR model performs best?
- Is more complexity (more residual blocks) always better?
- What's the trade-off between training time and accuracy?

### 3. Disease-Specific Analysis
- Which diseases benefit most from SR?
- Which diseases benefit least?
- Why might this be?

### 4. Practical Considerations
- Would you deploy this system on a real farm?
- What are the computational costs?
- How does this compare to just using higher resolution cameras?

### 5. Future Work
- How could we improve these results?
- What would you try next?
- How does this relate to the LASSR paper?

Write your answers in a markdown cell below:

## My Analysis

### 1. Overall Impact
[Your answer here]

### 2. Model Comparison
[Your answer here]

### 3. Disease-Specific Analysis
[Your answer here]

### 4. Practical Considerations
[Your answer here]

### 5. Future Work
[Your answer here]

---

## 🎓 Congratulations!

You've completed the core experiments. You now have:

✅ Trained multiple SR models  
✅ Evaluated their impact on disease classification  
✅ Analyzed the trade-offs  
✅ Generated publication-quality results  

### Next Steps

1. **Write your report**: Summarize methods, results, and conclusions
2. **Create presentation**: Share findings with mentor/class
3. **(Optional) Explore LASSR**: Read the paper and try to implement simplified version
4. **(Optional) Try extensions**: See README.md for ideas

### Resources

- **STUDENT_GUIDE.py**: Detailed instructions and troubleshooting
- **LASSR Paper**: arXiv:2010.06499
- **Your mentor**: Ask questions!

Great work! 🌟